In [ ]:
import sys
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from datetime import datetime

In [ ]:
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init("bronzeToSilverJob", {})

In [ ]:
# read bronze file dynamically
today = datetime.now()
day = today.strftime("%d")
month = today.strftime("%m")
year = today.strftime("%Y")

bronze_path = f"s3://job-market-tracker-hsg/bronze/year={year}/month={month}/day={day}/"

df = spark.read.option("multiline", "true").json(bronze_path)

In [ ]:
df_jobs = df.select(explode(col("jobs")).alias("job"))


In [ ]:
from pyspark.sql.functions import current_date

df_silver = df_jobs.select(
    col("job._id").alias("job_id"),
    col("job.company_name"),
    col("job.job_title"),
    col("job.job_type"),
    col("job.location_type"),
    to_date(col("job.date_posted")).alias("date_posted"),
    
    col("job.yoe_range.min").alias("min_yoe"),
    col("job.yoe_range.max").alias("max_yoe"),
    
    col("job.locations")[0]["city"].alias("location_city"),
    col("job.locations")[0]["region"].alias("location_region"),
    col("job.locations")[0]["country"].alias("location_country"),
    
    col("job.technologies"),
    col("job.skills"),
    col("job.description")
)

In [ ]:
df_silver = df_silver.dropDuplicates(["job_id"])
df_silver = df_silver.withColumn("year", lit(year)) \
    .withColumn("month", lit(month)) \
    .withColumn("day", lit(day))

In [ ]:
#write to silver dynamically
silver_path = f"s3://job-market-tracker-hsg/silver/"
df_silver.write \
    .mode("append") \
    .partitionBy("year", "month", "day") \
    .parquet(silver_path)

job.commit()